# Predicting δ13C from MIR spectra: findings
> What the cassava leaf data show about predicting leaf δ13C from mid-infrared spectra, and how to validate such models

**Work in progress.** The analysis isn't finished. The findings below come from a first round of models, and the growing conditions of the batches are still missing. Expect the results and conclusions to change.

## Summary

A PLS model on the MIR spectra reaches an R² of 0.77 on a random split stratified by batch. Batch identity alone reaches an R² of 0.68 on the same data. The spectra add little to what the batch already tells us.

The model can't predict the δ13C level of a batch it hasn't seen. Under leave-one-batch-out validation, its R² is -0.01.

Inside batches, the spectra carry real δ13C information. A local PLS model, calibrated on samples from one batch, explains 36% to 73% of the δ13C variation inside that batch. This holds for six of the seven batches. Batch 1592 shows no signal.

MIRS can rank samples within a trial when some samples from that trial have lab values. It can't replace lab measurements for a new trial.

## Why δ13C

Cassava is a C3 plant. During photosynthesis, its carbon-fixing enzyme, Rubisco, fixes CO2 that contains the light isotope 12C more readily than CO2 that contains 13C. How strongly the leaf favours 12C depends on the CO2 concentration inside the leaf, compared with the air outside.

A plant short of water closes its stomata to save water. The CO2 concentration inside the leaf then falls, and the leaf favours 12C less. Its δ13C rises, which means it becomes less negative.

Leaf δ13C therefore records the balance between carbon gain and water loss while the leaf grew. Plant scientists use it as a measure of water-use efficiency and of a plant's response to water stress. In wheat, for example, breeders have used it to select varieties that use water more efficiently.

Measuring δ13C needs isotope-ratio mass spectrometry, which is slow and costly for each sample. MIR spectra are fast and cheap to collect. A model that predicts δ13C from spectra would let breeders and physiologists screen many more plants.

## Data

The spectra file `specs_trim.csv` holds absorbance spectra of 1184 samples. Each spectrum has 1711 points from 651.8 to 3949.5 cm⁻¹, about 1.93 cm⁻¹ apart. The file `all_batches_wet_chem.csv` holds δ13C values for 1291 samples. 1158 samples appear in both files, and all the analyses use these 1158 samples.

Sample ids have the form `<batch>_<sample>`. Each of the seven batches is a set of plants grown under its own conditions and scanned at its own time, on the same spectrometer. The data don't record the conditions.

| Batch | Samples | Mean δ13C (‰) | Standard deviation (‰) |
|---|---|---|---|
| 1456 | 149 | -23.85 | 1.20 |
| 1586 | 168 | -28.87 | 1.00 |
| 1592 | 244 | -26.06 | 0.85 |
| 1597 | 97 | -28.11 | 0.78 |
| 1678 | 144 | -25.56 | 1.62 |
| 1722 | 177 | -28.54 | 1.11 |
| 1723 | 179 | -27.36 | 1.26 |

<img src="images/c13_by_batch.png" alt="Box plot of δ13C by batch" width="80%" style="display: block; margin: 0 auto;">

δ13C by batch. The batch means differ by up to 5 ‰.

## Batch structure

Batch explains 68% of the δ13C variance. This is the between-batch share of the variance (η²). The batch means span 5 ‰, from -28.87 ‰ in batch 1586 to -23.85 ‰ in batch 1456. Inside a batch, the standard deviation is 0.78 to 1.62 ‰.

The spectra also differ between batches. In a PCA of the raw spectra, the first component explains 92.6% of the variance. It reflects overall absorbance and baseline, and it doesn't correlate with δ13C (r = 0.00). SNV removes this effect, and the separate cluster of batch 1678 in the raw PCA disappears. After SNV and a Savitzky-Golay first derivative, five batches overlap in one core. Batches 1722 and 1723 lie along a separate arm. On the first component of these spectra, batch explains 80% of the variance (η² = 0.80). The arm comes from band shapes across the fingerprint region, with the largest loading at 1250 cm⁻¹. Inside batches 1722 and 1723, the position along the arm doesn't follow δ13C (r = -0.05 and 0.06).

The PCA plots and loadings are in the [EDA notebook](01_eda.ipynb).

<div class="column-page">
<img src="images/pca_by_batch.png" alt="PCA scores coloured by batch, for raw, SNV and SNV plus first-derivative spectra" width="100%">
</div>

First two PCA components of the raw spectra, the SNV spectra and the SNV first-derivative spectra (`snv_sg1`), coloured by batch. In the `snv_sg1` panel, the horizontal arm holds batches 1722 and 1723, and the vertical arm holds samples with water vapour.

## Data quality: water vapour

Some spectra contain water vapour lines. These are sharp, regular lines between about 3900 and 3550 cm⁻¹, with weaker ones between 1800 and 1300 cm⁻¹. They come from humidity in the beam path, not from the leaves.

We measured them with a water vapour index. The index takes each SNV spectrum, subtracts a Savitzky-Golay smoothed copy (11-point window), and returns the standard deviation of the residual between 3550 and 3900 cm⁻¹. It correlates with the second PCA component of the derivative spectra at r = 0.92. That component is therefore water vapour, not leaf chemistry.

| Batch | Samples above 3 times the median index | Share |
|---|---|---|
| 1456 | 0 | 0% |
| 1586 | 0 | 0% |
| 1592 | 27 | 11% |
| 1597 | 57 | 59% |
| 1678 | 4 | 3% |
| 1722 | 6 | 3% |
| 1723 | 2 | 1% |

The threshold of 3 times the median is arbitrary. It shows how far the problem spreads.

Four samples from batch 1722 stand apart in the PCA because of water vapour: 1722_092, 1722_120, 1722_138 and 1722_147. Sample 1722_138 also has a weaker band near 1250 cm⁻¹ than the rest of its batch.

Removing everything above 3550 cm⁻¹ changed the model results only slightly. The lines between 1800 and 1300 cm⁻¹ are still in the spectra, because trimming there would also remove the amide bands.

<img src="images/vapour_by_batch.png" alt="Box plot of the water vapour index by batch, on a log scale" width="80%" style="display: block; margin: 0 auto;">

Water vapour index by batch, on a log scale. Batch 1597 has the highest median.

## Validation

A random split puts samples from every batch into both training and test folds. The model then learns where each batch sits, and the test score rewards it for that. With 68% of the variance between batches, a random split gives a high R² even when the spectra say little about δ13C itself.

We used three scores:

- **Random split**, 5 folds stratified by batch. It tests predictions for new samples from batches the model has seen.
- **Leave one batch out**, 7 folds. It tests predictions for a batch the model has never seen.
- **Within-batch R²**. It centres the observed and predicted values on their batch means before computing R². A model that predicts only the batch means scores 0.

We also report the correlation r between observed and predicted values inside each batch. It measures only the order of the predictions, and ignores their level and spread. For ranking samples within a trial, r is the score that matters.

The random-split R² splits into these parts. Batch explains 0.68 of the variance. The remaining 0.32 lies inside batches, and the model explains 0.29 of it. Together, 0.68 + 0.29 × 0.32 = 0.77, which matches the random-split R².

We recommend reporting the within-batch R² and the leave-one-batch-out score next to any random-split R².

## Results

### Global PLS model

The pipeline applies SNV, a Savitzky-Golay first derivative (15-point window, cubic polynomial) and PLS regression without feature scaling. We chose the number of components by the lowest RMSE on the same folds that score the model. This makes all scores below slightly optimistic.

| Split | Components | R² | RMSE (‰) | Within-batch R² |
|---|---|---|---|---|
| Random | 17 | 0.77 | 0.97 | 0.29 |
| Leave one batch out | 20 | -0.01 | 2.01 | -0.19 |

When a batch is left out, the model pulls its predictions towards the mean of the other batches. Batch 1456 has the highest mean, and the model predicts it 3.45 ‰ too low.

| Held-out batch | Bias (‰) | Within-batch R² | r |
|---|---|---|---|
| 1456 | -3.45 | 0.13 | 0.42 |
| 1586 | 1.41 | 0.48 | 0.70 |
| 1592 | 0.45 | -1.39 | -0.03 |
| 1597 | 1.46 | -1.08 | 0.12 |
| 1678 | -1.38 | -0.33 | 0.04 |
| 1722 | 0.90 | -0.15 | 0.58 |
| 1723 | -0.78 | 0.27 | 0.61 |

The model ranks four new batches moderately well, with r from 0.42 to 0.70. It has no ranking skill for batches 1592, 1597 and 1678. No number of components between 3 and 20 gives consistent ranking on new batches. For single batches, the within-batch R² changes sign as the number of components changes. Removing the region above 3550 cm⁻¹ changes the results only slightly.

<img src="images/observed_predicted.png" alt="Observed against predicted δ13C for the full and trimmed spectra, on the random and leave-one-batch-out splits" width="100%" style="display: block; margin: 0 auto;">

Observed against predicted δ13C. The rows show the full and trimmed spectra. The columns show the random split and leave-one-batch-out. Under leave-one-batch-out, each batch's predictions move towards the mean of the other batches.

### Within-batch models

Two models learn only the variation inside batches. A local model is fitted on one batch, with 5-fold cross-validation inside it. The pooled model is fitted on all batches at once, after each batch is centred on its own mean spectrum and mean δ13C. Each fold computes the batch means from its training samples only. Both models need lab values from the batch they predict.

| Batch | Within-batch R², global | Within-batch R², local | Within-batch R², pooled | r, global | r, local | r, pooled |
|---|---|---|---|---|---|---|
| 1456 | 0.17 | 0.42 | 0.26 | 0.58 | 0.65 | 0.53 |
| 1586 | 0.57 | 0.73 | 0.66 | 0.77 | 0.86 | 0.81 |
| 1592 | -0.35 | 0.00 | -0.19 | 0.05 | 0.10 | 0.02 |
| 1597 | -0.49 | 0.36 | -0.13 | 0.25 | 0.66 | 0.33 |
| 1678 | 0.54 | 0.72 | 0.61 | 0.73 | 0.85 | 0.79 |
| 1722 | 0.30 | 0.62 | 0.51 | 0.69 | 0.79 | 0.77 |
| 1723 | 0.46 | 0.56 | 0.53 | 0.70 | 0.75 | 0.73 |

The global column is the random-split model. Local models do best in every batch. They use 2 to 15 components. The pooled model uses 12 components and reaches an overall R² of 0.81, an RMSE of 0.87 ‰ and a within-batch R² of 0.41.

Batch 1678 ranks well with its own local model (r = 0.85) and not at all when left out of the global model (r = 0.04). Its relationship between spectra and δ13C differs from the other batches. Batch 1597 shows the same pattern, less strongly.

Batch 1592 shows no signal, even with its own local model. It has the smallest δ13C spread of all batches, with a standard deviation of 0.85 ‰. Measurement error in the lab values could account for much of that spread.

The tables and plots are in the [PLS notebook](02_pls.ipynb).

## What this means for each use case

- **Ranking samples within a trial, with lab values for some of its samples.** This works. Local models reach r = 0.65 to 0.86 in six batches. The pooled model reaches r = 0.33 to 0.81 in the same six batches and needs fewer lab values per batch.
- **Ranking samples in a new trial with no lab values.** This works for some batches and fails for others. The data don't show in advance which group a new batch falls into.
- **Predicting the δ13C level of a new trial.** This doesn't work. The spectra don't carry the batch level in a form that transfers.

## Open questions and next steps

1. **Batch metadata.** The growing conditions, dates, genotypes and sample preparation of each batch could explain why 1678 and 1597 differ from the rest, and why 1722 and 1723 form their own spectral group. If genotypes repeat across batches, we can test whether the spectra rank genotypes the same way across conditions.
2. **Lab repeatability of δ13C.** This decides whether batch 1592 lacks signal or only lacks spread.
3. **Water vapour correction.** Subtracting a reference vapour spectrum would also remove the lines between 1800 and 1300 cm⁻¹. If the instrument software kept the single-beam spectra, its atmospheric compensation may be cleaner.
4. **Nested cross-validation.** Choosing the number of components in an inner loop gives unbiased scores.
5. **Other models.** Memory-based learning or Cubist are worth a bounded test after steps 1 to 4. They must beat PLS on the within-batch scores and on leave-one-batch-out ranking, not on the random split.

## Methods

- [00_data](00_data.ipynb) loads and aligns the two files.
- [01_eda](01_eda.ipynb) covers δ13C by batch, the PCA, the loadings and the water vapour index.
- [02_pls](02_pls.ipynb) covers the global, local and pooled PLS models.

The random split uses `StratifiedKFold(5, shuffle=True, random_state=0)`, stratified by batch. The local models use `KFold(5, shuffle=True, random_state=0)` inside each batch. Leave-one-batch-out uses `LeaveOneGroupOut`. All models use `soilspectfm` preprocessing and scikit-learn `PLSRegression(scale=False)`. The data files are not part of the repository.